# Tutorial 01: Time-Series Forecasting with PatchTST

This notebook demonstrates end-to-end forecasting with **PatchTST** — a SOTA 2023 model that applies patching and channel-independence to long-term industrial sensor forecasting.

**What you will learn:**
- How to build proper forecasting windows (input sequence → target sequence)
- How to use `ForecastingPipeline` with the updated metrics (RMSE, MAE, MAPE, SMAPE)
- How to visualise forecast vs. actual

**Dataset:** Synthetic SWaT-like data (51 sensor channels). No download required.

**Reference:** Nie et al., *A Time Series is Worth 64 Words: Long-term Forecasting with Transformers*, ICLR 2023.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

from models.patchtst import PatchTST
from forecasting.pipeline import ForecastingPipeline
from visualization.plot_utils import plot_multivariate_timeseries, plot_forecast_vs_actual

## 1. Build Forecasting Windows

We generate a synthetic multivariate time series and create **proper (X, Y) forecasting windows**:
- `X` — input sequence of length `seq_len = 96` steps
- `Y` — target sequence of length `pred_len = 24` steps immediately following X

This resolves the data reconciliation issue present in earlier versions of this tutorial.

In [ ]:
NUM_FEATURES = 51
SEQ_LEN = 96
PRED_LEN = 24
NUM_SAMPLES = 3000
BATCH_SIZE = 32

rng = np.random.default_rng(42)
# Synthetic sensor data with trend + seasonality
t = np.linspace(0, 10 * np.pi, NUM_SAMPLES + SEQ_LEN + PRED_LEN)
raw = (np.sin(t[:, None] + rng.uniform(0, np.pi, NUM_FEATURES)) 
       + 0.3 * rng.standard_normal((len(t), NUM_FEATURES))).astype(np.float32)

# Build (X, Y) pairs
xs = np.stack([raw[i : i + SEQ_LEN] for i in range(NUM_SAMPLES)])       # (N, 96, 51)
ys = np.stack([raw[i + SEQ_LEN : i + SEQ_LEN + PRED_LEN] for i in range(NUM_SAMPLES)])  # (N, 24, 51)

print(f"X shape: {xs.shape}  →  (samples, seq_len, features)")
print(f"Y shape: {ys.shape}  →  (samples, pred_len, features)")

# Visualise a snippet of the raw data
fig = plot_multivariate_timeseries(raw[:200], title="Synthetic SWaT-like Sensor Data (first 200 steps)", max_subplots=4)
plt.show()

In [ ]:
split = int(0.8 * NUM_SAMPLES)
X_train, Y_train = torch.tensor(xs[:split]), torch.tensor(ys[:split])
X_val,   Y_val   = torch.tensor(xs[split:]), torch.tensor(ys[split:])

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(TensorDataset(X_val,   Y_val),   batch_size=BATCH_SIZE, shuffle=False, drop_last=True)

print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

## 2. Initialise PatchTST

PatchTST processes each sensor channel independently:
1. Splits the input sequence into overlapping patches of length 16 with stride 8
2. Projects patches to a `d_model`-dimensional embedding
3. Applies a Transformer encoder
4. Flattens and projects to the forecast horizon

In [ ]:
model = PatchTST(
    num_features=NUM_FEATURES,
    seq_len=SEQ_LEN,
    pred_len=PRED_LEN,
    patch_len=16,
    stride=8,
    d_model=64,
    nhead=4,
    num_layers=2,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"PatchTST total parameters: {n_params:,}")

# Quick shape check
dummy = torch.randn(2, SEQ_LEN, NUM_FEATURES)
out = model(dummy)
print(f"Input shape:  {dummy.shape}  →  (batch, seq_len, features)")
print(f"Output shape: {out.shape}   →  (batch, pred_len, features)")

## 3. Train with ForecastingPipeline

The pipeline handles:
- Adam optimiser with ReduceLROnPlateau scheduling
- Per-epoch RMSE, MAE, MAPE, SMAPE reporting
- Best-checkpoint saving by validation RMSE

In [ ]:
pipeline = ForecastingPipeline(
    model=model,
    learning_rate=1e-3,
    model_name="PatchTST",
    dataset_name="SWaT_dummy",
    pred_len=PRED_LEN,
)

metrics = pipeline.fit(train_loader, val_loader, epochs=10)

## 4. Visualise Results

In [ ]:
# Collect predictions from the validation set
model.eval()
device = next(model.parameters()).device
preds, targets = [], []

with torch.no_grad():
    for bx, by in val_loader:
        p = model(bx.to(device))
        preds.append(p.cpu().numpy())
        targets.append(by.numpy())

preds   = np.concatenate(preds,   axis=0)  # (N, pred_len, features)
targets = np.concatenate(targets, axis=0)

# Plot first sensor channel
fig = plot_forecast_vs_actual(
    targets=targets,
    predictions=preds,
    feature_idx=0,
    title=f"PatchTST — Sensor 0  |  pred_len={PRED_LEN}",
    pred_len=PRED_LEN,
)
plt.show()

print(f"\nFinal validation metrics:")
print(f"  RMSE  = {metrics.rmse:.4f}")
print(f"  MAE   = {metrics.mae:.4f}")
print(f"  MAPE  = {metrics.mape:.2f}%")
print(f"  SMAPE = {metrics.smape:.2f}%")